# Практика § 4.5: обучающие пары

Работаем с придуманными двумерными векторами. Они не являются выводом готового CLIP.

## Матрица в двух направлениях

Сначала выберите лучший снимок для каждого текста и лучшую подпись для каждого снимка на бумаге.

In [ ]:
import numpy as np
images = np.array([[0.8,0.6], [0.,1.]])
texts = np.array([[1.,0.], [0.,1.]])
scores = images @ texts.T
print(scores)
print("Лучший текст по снимкам:", scores.argmax(axis=1))
print("Лучший снимок по текстам:", scores.argmax(axis=0))

## Перепутанные подписи

Поменяйте порядок подписей, оставив изображения прежними. Почему диагональ больше не соответствует правильным парам?

In [ ]:
swapped = texts[::-1]
print(images @ swapped.T)

## Ошибка двух направлений

Используйте softmax отдельно по строкам и по столбцам. Затем сравните ошибку при верном и перепутанном порядке.

In [ ]:
def softmax_rows(m):
    shifted = m - m.max(axis=1, keepdims=True)
    values = np.exp(shifted)
    return values / values.sum(axis=1, keepdims=True)
def pair_loss(m):
    row = -np.log(np.diag(softmax_rows(m))).mean()
    col = -np.log(np.diag(softmax_rows(m.T))).mean()
    return (row+col)/2
print("Верные пары:", round(pair_loss(scores),3))
print("Перепутанные:", round(pair_loss(images @ swapped.T),3))

## Нет подходящего снимка

Добавьте третий запрос, которому не соответствует ни один снимок. Ранжирование всё равно вернёт лидера. Объясните, какие размеченные примеры нужны для настройки отказа от ответа.

## Самопроверка




### 1. Что выдают два энкодера CLIP?

A. Два списка файлов

B. Сравнимые векторы изображения и текста

C. Рамку и маску

D. Следующий токен и метку




### 2. Что означает диагональ обучающей матрицы при правильном порядке данных?

A. Случайные ответы

B. Соответствующие пары изображения и подписи

C. Только отрицательные пары

D. Площади изображений




### 3. Для поиска снимка по одной подписи что сравнивают?

A. Строки одного столбца

B. Столбцы одной строки

C. Только диагональ

D. Длины имён файлов




### 4. Для выбора подписи к одному снимку что сравнивают?

A. Строки одного столбца

B. Столбцы одной строки

C. Только минимум

D. Число пикселей




### 5. Почему косинус 0,8 не означает «80 % вероятности правды»?

A. Он служит мерой близости векторов, а не калиброванной вероятностью истины

B. Он всегда отрицателен

C. Он получен из размера файла

D. Он равен числу классов




### 6. Какие два направления использует контрастивная обучающая задача?

A. Класс→пиксель и пиксель→класс

B. Изображение→текст и текст→изображение

C. Вчера→сегодня и сегодня→завтра

D. Обучение→тест и тест→обучение




### 7. Что происходит при перепутанных подписях в обучающей таблице?

A. Сеть получает неверный сигнал о том, какие пары надо сближать

B. Все оценки обнуляются автоматически

C. Снимки исчезают

D. Матрица перестаёт быть квадратной




### 8. Что выдаст поиск, если подходящего снимка нет в коллекции и правила отказа нет?

A. Ничего

B. Кандидата с наибольшей оценкой среди неподходящих

C. Новое изображение

D. Правильный ответ из интернета




### 9. Почему одинаковая подпись для многих разных изображений осложняет обучение?

A. Она сообщает мало различий между ними

B. Слишком короткий файл нельзя открыть

C. У каждого изображения должна быть отдельная модель

D. Косинус нельзя вычислить




### 10. Что нужно для проверки правила «ничего не найдено»?

A. Только один удачный запрос

B. Запросы с подходящими и неподходящими снимками и отдельная итоговая проверка

C. Только английские слова

D. Одна диагональная матрица




## Подсказки и решения

### Вопрос 1

**Подсказка**


Что сравнивают скалярным произведением?



**Решение**


**B.** Оба входа превращаются в векторы общего пространства.



### Вопрос 2

**Подсказка**


Как упорядочены строки и столбцы?



**Решение**


**B.** Пара с одним индексом относится к исходному соответствию.



### Вопрос 3

**Подсказка**


Фиксируйте текстовый запрос.



**Решение**


**A.** При фиксированной подписи сравнивают оценки всех изображений в её столбце.



### Вопрос 4

**Подсказка**


Фиксируйте изображение.



**Решение**


**B.** В строке одного изображения сравнивают все подписи.



### Вопрос 5

**Подсказка**


Что именно вычисляет скалярное произведение?



**Решение**


**A.** Сходство векторов не является проверкой истинности.



### Вопрос 6

**Подсказка**


Прочитайте строки и столбцы матрицы.



**Решение**


**B.** Правильную пару выделяют и по строке, и по столбцу.



### Вопрос 7

**Подсказка**


Что считается правильной парой после перестановки?



**Решение**


**A.** Ошибочная разметка сближает несоответствующие входы.



### Вопрос 8

**Подсказка**


Может ли сортировка оставить список пустым сама?



**Решение**


**B.** Ранжирование всё равно имеет первый элемент.



### Вопрос 9

**Подсказка**


Какие альтернативы в пакете становятся похожими?



**Решение**


**A.** Общие подписи затрудняют различение пар.



### Вопрос 10

**Подсказка**


Как увидеть ложный отказ и ложное совпадение?



**Решение**


**B.** Нужны оба вида запросов и проверка вне данных подбора.



